# Import libraries

In [1]:
import pandas as pd
import os
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import threading
from typing import Tuple
from tqdm import tqdm

import sys
sys.path.append('..')
from utils.audio_util import convert_mp3_to_flac, resample_audios, trim_silence_with_vad, normalize_audio_files
from utils.file_util import recursive_copy

# Moving files to new directory

In [2]:
df = pd.read_csv("../data/raw/thai-central/thai-central_mapping.csv")

In [3]:
AUDIO_BASE_DIR = "../data/raw/thai-central/audio_v2"
DEST_DIR = "../data/converted/thai-central-to-vctk"
AUDIO_DEST_DIR = os.path.join(DEST_DIR, "wav16")
TXT_DEST_DIR = os.path.join(DEST_DIR, "txt")

In [4]:
# Add full path column
df['full_path'] = df['public_name'].apply(lambda x: os.path.join(AUDIO_BASE_DIR, x))

# Filter existing files
df_filtered = df[df['full_path'].apply(os.path.exists)].copy()

# Count files per speaker
speaker_counts = df_filtered['speaker_id'].value_counts()
valid_speakers = speaker_counts[speaker_counts >= 100].index

# Filter speakers with >= 100 files
df_filtered = df_filtered[df_filtered['speaker_id'].isin(valid_speakers)]

In [5]:
# Create new speaker ID mapping
sorted_speakers = speaker_counts[speaker_counts >= 100].sort_values().index
speaker_mapping = {
    spk: f'tc{i+1:04d}' 
    for i, spk in enumerate(sorted_speakers)
}

# Add new speaker ID column
df_filtered['new_speaker_id'] = df_filtered['speaker_id'].map(speaker_mapping)
df_filtered

,speaker_id,original_name,public_name,full_path,new_speaker_id
3,spk-560353791001250275300,60f7422cf22f3d24deb858fd_1627814081674.wav,train_audio02/thai-central_000003.mp3,../data/raw/thai-central/audio_v2/train_audio0...,tc0776
5,spk8789214662434341888000,611bdfa7cf5abe28e3190dab_1637558034204.wav,train_audio12/thai-central_000005.mp3,../data/raw/thai-central/audio_v2/train_audio1...,tc0675
6,spk2272676593238898160000,6107d2adfb309b3360224bb3_1637129050946.wav,train_audio06/thai-central_000006.mp3,../data/raw/thai-central/audio_v2/train_audio0...,tc0571
7,spk-659303002028011164800,60f7422cf22f3d24deb85922_1628183509033.wav,train_audio01/thai-central_000007.mp3,../data/raw/thai-central/audio_v2/train_audio0...,tc0898
8,spk5308879110509872434000,60f7422cf22f3d24deb858ff_1628702261034.wav,train_audio02/thai-central_000008.mp3,../data/raw/thai-central/audio_v2/train_audio0...,tc0965
...,...,...,...,...,...
433806,spk-717737201021391723000,611be1b6cf5abe28e3191fe0_1636618869118.wav,train_audio05/thai-central_433806.mp3,../data/raw/thai-central/audio_v2/train_audio0...,tc0649
433808,spk8339279944661639402000,60f7422cf22f3d24deb85917_1629169814235.wav,train_audio02/thai-central_433808.mp3,../data/raw/thai-central/audio_v2/train_audio0...,tc0923
433811,spk7617813596557544310000,6107d2adfb309b3360224ba8_1637550218939.wav,train_audio12/thai-central_433811.mp3,../data/raw/thai-central/audio_v2/train_audio1...,tc0656
433812,spk-581523479174861464600,611be1b6cf5abe28e3191fd3_1629719575649.wav,train_audio04/thai-central_433812.mp3,../data/raw/thai-central/audio_v2/train_audio0...,tc0683


In [6]:
df_train = pd.read_csv("../data/raw/thai-central/train.csv")
df_dev = pd.read_csv("../data/raw/thai-central/dev.csv")

df_all = pd.concat([df_train, df_dev], ignore_index=True)
df_all['sentence'] = df_all['sentence'].apply(lambda x: "".join(x.split()))
df_all['audio'] = df_all['audio'].apply(lambda x: os.path.join(AUDIO_BASE_DIR, x))
df_all

,utterance,sentence,audio
0,thai-central_000000,ทีมจากอิสราเอลไม่ควรได้เป็นเจ้าบ้านในเกมยูฟ่าคัพ,../data/raw/thai-central/audio_v2/train_audio1...
1,thai-central_000001,แต่พอไหมอะไรคือแต้อีบ็อบฮ่าฮ่ากูพิมพ์ผิดไหมล่ะ...,../data/raw/thai-central/audio_v2/train_audio0...
2,thai-central_000003,ทุกสิ่งทุกอย่างจะราบรื่น,../data/raw/thai-central/audio_v2/train_audio0...
3,thai-central_000005,เร็วหันมองเวลาตั้งกระทู้,../data/raw/thai-central/audio_v2/train_audio1...
4,thai-central_000006,มีขนาดหนาและใหญ่กว่าเกร็ดปลาทั่วไปจนเหมือนเครื...,../data/raw/thai-central/audio_v2/train_audio0...
...,...,...,...
341134,thai-central_433292,มีของทั้งหมดเป็นจำนวนหนึ่งหมื่นหนึ่งพันกระป๋องค่ะ,../data/raw/thai-central/audio_v2/dev_audio00/...
341135,thai-central_433304,บ้านงิ้วงามหมู่สี่มีอาณาเขตติดต่อกับหมู่บ้านใก...,../data/raw/thai-central/audio_v2/dev_audio00/...
341136,thai-central_433457,กองทัพเรือหมายถึงกองกำลังทางทหารที่ปฏิบัติการท...,../data/raw/thai-central/audio_v2/dev_audio00/...
341137,thai-central_433700,กรมอู่ทหารเรือ,../data/raw/thai-central/audio_v2/dev_audio00/...


In [7]:
audio2sentence = dict(zip(df_all['audio'], df_all['sentence']))

In [8]:
# Thread-safe set for character collection
all_chars = set()
chars_lock = threading.Lock()

# Thread-safe list for tracking skipped files
skip_files = []
skip_lock = threading.Lock()

def process_file_pair(args: Tuple[str, str, str, str]) -> None:
    """Process a single pair of audio and text files"""
    speaker_id, src_path, dest_audio_path, dest_txt_path = args
    try:
        # Create speaker directories
        speaker_wav_dir = os.path.join(AUDIO_DEST_DIR, speaker_id)
        speaker_txt_dir = os.path.join(TXT_DEST_DIR, speaker_id)
        os.makedirs(speaker_wav_dir, exist_ok=True)
        os.makedirs(speaker_txt_dir, exist_ok=True)
        
        # Process audio
        dest_filename = os.path.splitext(os.path.basename(dest_audio_path))[0] + '_mic1.flac'
        dest_path = os.path.join(speaker_wav_dir, dest_filename)
        
        if not convert_mp3_to_flac(src_path, dest_path):
            raise Exception("Failed to convert audio")
        
        # Create empty text file and collect characters
        base_filename = os.path.splitext(dest_filename)[0]
        txt_filename = f"{base_filename}.txt"
        txt_path = os.path.join(speaker_txt_dir, txt_filename)
        
        # In this case we're creating empty text files
        # Modify this part if you need to process actual text content
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(audio2sentence[src_path])

        # Collect characters
        with chars_lock:
            all_chars.update(audio2sentence[src_path])
            
    except Exception as e:
        print(f"Error processing file {src_path}: {e}")
        with skip_lock:
            skip_files.append(src_path)

# Remove existing directories if they exist
if os.path.exists(DEST_DIR):
    print("Clearing destination folder")
    shutil.rmtree(DEST_DIR)

# Create necessary directories
os.makedirs(AUDIO_DEST_DIR, exist_ok=True)
os.makedirs(TXT_DEST_DIR, exist_ok=True)

# Create processing arguments
process_args = [
    (row['new_speaker_id'], row['full_path'], 
        os.path.join(AUDIO_DEST_DIR, row['new_speaker_id'], os.path.basename(row['public_name'])),
        os.path.join(TXT_DEST_DIR, row['new_speaker_id'], os.path.basename(row['public_name'])))
    for _, row in df_filtered.iterrows()
]

# Process files in parallel with progress bar
max_workers = os.cpu_count()
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    list(tqdm(
        executor.map(process_file_pair, process_args),
        total=len(process_args),
        desc=f"Processing files (using {max_workers} workers)"
    ))

# Print results
print(f"Processed {len(df_filtered) - len(skip_files)} file pairs")
print(f"Skipped {len(skip_files)} pairs")
print(f"Unique characters found: {''.join(sorted(all_chars))}")

Processing files (using 16 workers): 100%|██████████| 225028/225028 [53:34<00:00, 70.00it/s] 

Processed 225028 file pairs
Skipped 0 pairs
Unique characters found: กขคฆงจฉชซฌญฎฏฐฑฒณดตถทธนบปผฝพฟภมยรฤลวศษสหฬอฮฯะัาำิีึืุูเแโใไ็่้๊๋์


# Save metadata

In [9]:
DEST_DIR = Path(DEST_DIR)

# Write character files
sorted_chars = sorted(all_chars)
with open(DEST_DIR / 'all_chars_unicode.txt', 'w') as f:
   f.write(''.join(c.encode('unicode_escape').decode('ascii') for c in sorted_chars))
   
with open(DEST_DIR / 'all_chars.txt', 'w') as f:
   f.write(''.join(sorted_chars))

# Resample, trim, and normalize audio

In [10]:
# Create destination directory if it doesn't exist
os.makedirs("../data/converted/thai-central-to-vctk/wav16_silence_trimmed", exist_ok=True)

# Copy all files from wav16 to wav16_silence_trimmed
src_dir = "../data/converted/thai-central-to-vctk/wav16"
dst_dir = "../data/converted/thai-central-to-vctk/wav16_silence_trimmed"

recursive_copy(src_dir, dst_dir)

In [11]:
# Resample all files in wav16_silence_trimmed to 16kHz
SAMPLE_RATE = 16000
NUM_RESAMPLE_THREADS = 8

resample_audios(
  input_folders=dst_dir,
  file_ext="flac",
  sample_rate=SAMPLE_RATE,
  n_jobs=NUM_RESAMPLE_THREADS
)

Resampling the audio files...
Found 225028 files...


100%|██████████| 225028/225028 [03:25<00:00, 1092.91it/s]


Done !


In [12]:
# Trim silence at the beginning and end of each audio file
trim_silence_with_vad(
  input_folder=dst_dir,
  file_extension="flac",
)

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /home/ming/.cache/torch/hub/master.zip


Found 225028 .flac files to process


Processing files:   2%|▏         | 4005/225028 [06:58<9:38:20,  6.37it/s] 

> The file ../data/converted/thai-central-to-vctk/wav16_silence_trimmed/tc0171/thai-central_409359_mic1.flac probably does not have speech please check it !!


Processing files:   2%|▏         | 4155/225028 [07:16<7:50:27,  7.82it/s] 

...

> The file ../data/converted/thai-central-to-vctk/wav16_silence_trimmed/tc0957/thai-central_040425_mic1.flac probably does not have speech please check it !!


Processing files:  98%|█████████▊| 219643/225028 [6:48:25<08:03, 11.15it/s]

> The file ../data/converted/thai-central-to-vctk/wav16_silence_trimmed/tc0616/thai-central_141438_mic1.flac probably does not have speech please check it !!


Processing files: 100%|██████████| 225028/225028 [6:57:21<00:00,  8.99it/s]



Processing complete

Found 523 files with no speech. List saved to ../data/converted/thai-central-to-vctk/no_speech_files.txt


In [13]:
# Normalize the volume of all audio files to -27dB
normalize_audio_files(dst_dir, "flac", -27, 16000)

Normalizing audio files: 100%|██████████| 225028/225028 [14:23:45<00:00,  4.34it/s]  
